# 04 -- E5 и адаптер запросов

Проверяю семантический поиск, который может находить похожие услуги без совпадения слов.
Использую multilingual-e5-small и небольшой адаптер запросов. Весь encoder не дообучаю.
Сравниваю оба варианта на dev; известные положительные пары base исключаю из негативов.

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

de_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
import nbformat

# Подключаю только ячейки с функциями; расчеты других ноутбуков не запускаю.
for de_file in ['02_feature_engineering.ipynb', '03_baselines.ipynb']:
    for de_cell in nbformat.read(de_root / 'notebooks' / de_file, as_version=4).cells:
        if 'definitions' in de_cell.metadata.get('tags', []):
            exec(de_cell.source, globals())
de_out = de_root / 'outputs'
de_seed = 143


In [2]:
import os
from huggingface_hub import snapshot_download

de_manifest = json.loads((de_out / 'models/dense_manifest.json').read_text())
de_model_path = de_out / 'models/multilingual-e5-small'
assert de_manifest.get('revision')
if not (de_model_path / 'model.safetensors').exists():
    snapshot_download(
        de_manifest['model'],
        revision=de_manifest['revision'],
        local_dir=de_model_path,
        allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', 'README.md'],
        ignore_patterns=['onnx/*', 'openvino/*'],
        max_workers=3,
    )


/Users/dvsorochan/.codex/.chatgpt-projects/g-p-6ab7bad8eeb481919ab7a788d369d736/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import gc
import torch
from sentence_transformers import SentenceTransformer

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_num_threads(4)
de_device = (
    'mps'
    if torch.backends.mps.is_available()
    else ('cuda' if torch.cuda.is_available() else 'cpu')
)
de_model = SentenceTransformer(
    str(de_out / 'models/multilingual-e5-small'),
    device=de_device,
    local_files_only=True,
)
de_model.max_seq_length = 128
de_items = pd.read_parquet(
    de_out / 'cache/items.parquet',
    columns=['item_id', 'dense_text', 'item_location_id'],
)
de_eval = pd.read_parquet(de_out / 'cache/eval_queries.parquet')
de_benchmark = pd.read_parquet(de_out / 'cache/benchmark_queries.parquet')
de_base = pd.read_parquet(de_out / 'cache/base_pairs.parquet')
de_sample = de_base.sample(
    n=min(18000, len(de_base)), random_state=de_seed
).reset_index(drop=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9893.05it/s]

In [4]:
de_train_queries = de_sample.drop_duplicates('query_id')
de_queries = (
    pd.concat([de_eval, de_benchmark, de_train_queries], ignore_index=True)
    .drop_duplicates('query_id')
    .reset_index(drop=True)
)
print('device:', de_device, 'items:', len(de_items), 'queries:', len(de_queries))


device: mps items: 515895 queries: 28188


## Embeddings с возобновлением
Кэш привязан к данным, model revision, порядку item_id и длине 128 tokens.
Каждый завершенный chunk записывается на диск. При аварии последний незавершенный chunk пересчитывается.
Ограничение 128 -- вычислительный компромисс; весь текст по-прежнему доступен lexical и pairwise признакам.

In [5]:
import hashlib

de_signature = hashlib.sha256(
    (
        sha256_file(de_out / 'data_manifest.json')
        + (de_out / 'models/dense_manifest.json').read_text()
        + '|128|'
        + '|'.join(de_items.item_id)
        + hashlib.sha256(
            pd.util.hash_pandas_object(de_items.dense_text, index=False).values.tobytes()
        ).hexdigest()
    ).encode()
).hexdigest()
de_checkpoint = de_out / 'cache/dense_progress.json'
de_vec_path = de_out / 'cache/item_embeddings.npy'
de_progress = json.loads(de_checkpoint.read_text()) if de_checkpoint.exists() else {}
if de_progress.get('signature') == de_signature and de_vec_path.exists():
    de_vectors = np.lib.format.open_memmap(de_vec_path, mode='r+')
    de_start = int(de_progress['completed'])
else:
    de_vectors = np.lib.format.open_memmap(
        de_vec_path, mode='w+', dtype='float32', shape=(len(de_items), 384)
    )
    de_start = 0
de_clock = time.monotonic()
for de_startrow in range(de_start, len(de_items), 2048):
    de_end = min(de_startrow + 2048, len(de_items))
    de_vectors[de_startrow:de_end] = de_model.encode(
        ['passage: ' + s for s in de_items.dense_text.iloc[de_startrow:de_end]],
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    de_vectors.flush()
    write_json(
        {'signature': de_signature, 'completed': de_end, 'items': len(de_items)},
        de_checkpoint,
    )
    if de_startrow % 20480 == 0:
        print(
            de_end,
            '/',
            len(de_items),
            'elapsed',
            round(time.monotonic() - de_clock),
            flush=True,
        )


309248 / 515895 elapsed 39


329728 / 515895 elapsed 87


350208 / 515895 elapsed 123


370688 / 515895 elapsed 154


391168 / 515895 elapsed 185


...

432128 / 515895 elapsed 247


452608 / 515895 elapsed 278


473088 / 515895 elapsed 309


493568 / 515895 elapsed 340


514048 / 515895 elapsed 371


In [6]:
de_query_vectors = de_model.encode(
    ['query: ' + s for s in de_queries.query_text],
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
np.save(de_out / 'cache/query_embeddings.npy', de_query_vectors)
de_queries[['query_id']].to_parquet(
    de_out / 'cache/embedding_query_ids.parquet', index=False
)
write_json(
    {
        'signature': de_signature,
        'max_seq_length': 128,
        'dimension': 384,
        'device': de_device,
        'item_encoding_seconds_this_run': time.monotonic() - de_clock,
    },
    de_out / 'validation/dense_runtime.json',
)
del de_model
gc.collect()
if de_device == 'mps':
    torch.mps.empty_cache()


Batches:   0%|          | 0/441 [00:00<?, ?it/s]

Batches:   0%|          | 2/441 [00:00<00:25, 17.10it/s]

Batches:   1%|          | 5/441 [00:00<00:18, 23.27it/s]

Batches:   2%|▏         | 8/441 [00:00<00:16, 25.93it/s]

Batches:   3%|▎         | 12/441 [00:00<00:15, 28.53it/s]

...

Batches:  93%|█████████▎| 412/441 [00:08<00:00, 73.29it/s]

Batches:  95%|█████████▌| 420/441 [00:08<00:00, 74.84it/s]

Batches:  97%|█████████▋| 428/441 [00:08<00:00, 75.57it/s]

Batches:  99%|█████████▉| 437/441 [00:08<00:00, 75.85it/s]

Batches: 100%|██████████| 441/441 [00:08<00:00, 52.33it/s]

In [7]:
de_qpos = dict(zip(de_queries.query_id, range(len(de_queries))))
de_eval_vectors = de_query_vectors[[de_qpos[q] for q in de_eval.query_id]]
de_candidates = dense_search(de_eval_vectors, de_vectors, de_eval, de_items, k=120)
de_candidates.to_parquet(de_out / 'cache/candidates_dense.parquet', index=False)
de_truth_all = json.loads((de_out / 'cache/truth.json').read_text())
de_dev = de_eval[de_eval.split.eq('dev')]
de_truth = {q: de_truth_all[q] for q in de_dev.query_id}
de_scores = []
for de_source, de_group in de_candidates[
    de_candidates.query_id.isin(de_dev.query_id)
].groupby('source'):
    de_pred = predictions_from_frame(
        de_group.sort_values(['query_id', 'rank']), de_items, k=100
    )
    de_result = recall_table(de_pred, de_truth).merge(
        de_dev[['query_id', 'regime']], on='query_id'
    )
    for de_regime, de_part in de_result.groupby('regime'):
        de_scores.append(
            {
                'source': de_source,
                'regime': de_regime,
                **de_part.filter(like='recall@').mean().to_dict(),
            }
        )
display(pd.DataFrame(de_scores))
pd.DataFrame(de_scores).to_csv(de_out / 'validation/dense_dev.csv', index=False)


,source,regime,recall@1,recall@10,recall@20,recall@50,recall@100
0,dense,cold,0.021333,0.111875,0.152792,0.242236,0.304347
1,dense,warm,0.015500,0.089167,0.123667,0.198643,0.271286
2,dense_local,cold,0.181936,0.479744,0.564847,0.630472,0.679014
3,dense_local,warm,0.237833,0.543012,0.617548,0.696012,0.725548


## Сложные неразмеченные кандидаты

Для обучающих запросов выбираю близкие объявления, исключая **все известные base-позитивы
нормализованного текста**, включая другие фильтры. Holdout не используется даже для очистки негативов.
Это снижает false negatives, но не устраняет их: других релевантных услуг может быть много.
Сохраняю их как `unlabeled`, а не как достоверные отрицательные примеры.

Далее обучаю небольшой residual query adapter поверх замороженного E5. Это supervised dense adaptation,
не полное дообучение transformer. Item-векторы остаются неизменными; итоговый выигрыш проверяю на dev.

In [8]:
de_mining_queries = de_train_queries.head(3000)
de_mining_vectors = de_query_vectors[[de_qpos[q] for q in de_mining_queries.query_id]]
de_mined = dense_search(
    de_mining_vectors, de_vectors, de_mining_queries, de_items, k=30
)
de_known = de_base.groupby('text_key').item_id.agg(set).to_dict()
de_text_map = de_mining_queries.set_index('query_id').text_key.to_dict()
de_item_ids = de_items.item_id.to_numpy()
de_mined['item_id'] = de_item_ids[de_mined.pos]
de_mined = de_mined[
    [
        i not in de_known.get(de_text_map[q], set())
        for q, i in zip(de_mined.query_id, de_mined.item_id)
    ]
]
de_mined = de_mined.sort_values(['query_id', 'rank', 'pos']).drop_duplicates(
    ['query_id', 'pos']
)
de_mined['label_status'] = 'retrieved_unlabeled'
de_mined.to_parquet(de_out / 'validation/hard_negatives.parquet', index=False)
assert all(
    i not in de_known.get(de_text_map[q], set())
    for q, i in zip(de_mined.query_id, de_mined.item_id)
)
display(de_mined.head(10))


,query_id,pos,source,rank,score,item_id,label_status
138501,00157355366e989845d345a8,428965,dense,1,0.913723,d4ba9748abd0d5dd,retrieved_unlabeled
138532,00157355366e989845d345a8,181449,dense_local,2,0.901188,5a01af0fffb587e0,retrieved_unlabeled
138502,00157355366e989845d345a8,333132,dense,2,0.907190,a521239c696bd15d,retrieved_unlabeled
138504,00157355366e989845d345a8,80171,dense,4,0.903260,27adff84ffa71a99,retrieved_unlabeled
138534,00157355366e989845d345a8,289073,dense_local,4,0.899470,8f277027b5ed6d2b,retrieved_unlabeled
138535,00157355366e989845d345a8,42377,dense_local,5,0.898707,14dda7e71a7606dd,retrieved_unlabeled
138506,00157355366e989845d345a8,480059,dense,6,0.903225,ee26091fa170daa1,retrieved_unlabeled
138507,00157355366e989845d345a8,216103,dense,7,0.902758,6b18c9586c4b69c7,retrieved_unlabeled
138537,00157355366e989845d345a8,413920,dense_local,7,0.897179,cd4646fec4815470,retrieved_unlabeled
138538,00157355366e989845d345a8,208673,dense_local,8,0.897015,6771910b52a2e447,retrieved_unlabeled


In [9]:
# Адаптер близок к identity; предметная специализация не должна разрушать pretrained пространство.
torch.manual_seed(de_seed)
de_adapter = torch.nn.Linear(384, 384, bias=False)
torch.nn.init.zeros_(de_adapter.weight)
de_optimizer = torch.optim.AdamW(de_adapter.parameters(), lr=2e-4, weight_decay=0.01)
de_item_pos = dict(zip(de_items.item_id, range(len(de_items))))
de_train_x = torch.tensor(de_query_vectors[[de_qpos[q] for q in de_sample.query_id]])
de_train_y = torch.tensor(
    np.array(de_vectors[[de_item_pos[i] for i in de_sample.item_id]])
)
de_neg_by_query = de_mined.groupby('query_id').pos.first().to_dict()
de_rng = np.random.default_rng(de_seed)
de_losses = []
for de_epoch in range(3):
    de_epoch_losses = []
    for de_batch in np.array_split(
        de_rng.permutation(len(de_sample)), max(1, len(de_sample) // 128)
    ):
        de_x = de_train_x[de_batch]
        de_z = torch.nn.functional.normalize(de_x + 0.1 * de_adapter(de_x), dim=1)
        de_y = de_train_y[de_batch]
        de_logits = de_z @ de_y.T / 0.05
        # Остальные известные позитивы запроса не становятся in-batch negatives.
        de_batch_items = de_sample.item_id.iloc[de_batch].tolist()
        de_mask = np.array(
            [
                [item in de_known.get(text, set()) for item in de_batch_items]
                for text in de_sample.text_key.iloc[de_batch]
            ]
        )
        np.fill_diagonal(de_mask, False)
        de_logits = de_logits.masked_fill(torch.tensor(de_mask), -1e4)
        de_neg_positions = [
            de_neg_by_query.get(q, -1) for q in de_sample.query_id.iloc[de_batch]
        ]
        de_neg = torch.tensor(np.array(de_vectors[np.maximum(de_neg_positions, 0)]))
        de_neg_logits = (de_z * de_neg).sum(1, keepdim=True) / 0.05
        de_neg_logits[torch.tensor(np.array(de_neg_positions) < 0)] = -1e4
        de_logits = torch.cat([de_logits, de_neg_logits], dim=1)
        de_loss = torch.nn.functional.cross_entropy(
            de_logits, torch.arange(len(de_batch))
        )
        de_optimizer.zero_grad()
        de_loss.backward()
        de_optimizer.step()
        de_epoch_losses.append(float(de_loss.detach()))
    de_losses.append({'epoch': de_epoch + 1, 'loss': np.mean(de_epoch_losses)})
    print(de_losses[-1], flush=True)


{'epoch': 1, 'loss': np.float64(2.7073663209165844)}


{'epoch': 2, 'loss': np.float64(1.8429417073726655)}


{'epoch': 3, 'loss': np.float64(1.7645777583122253)}


In [10]:
torch.save(de_adapter.state_dict(), de_out / 'models/query_adapter.pt')
with torch.no_grad():
    de_adapted = torch.nn.functional.normalize(
        torch.tensor(de_query_vectors)
        + 0.1 * de_adapter(torch.tensor(de_query_vectors)),
        dim=1,
    ).numpy()
np.save(de_out / 'cache/query_embeddings_adapted.npy', de_adapted)
de_adapted_candidates = dense_search(
    de_adapted[[de_qpos[q] for q in de_eval.query_id]],
    de_vectors,
    de_eval,
    de_items,
    k=120,
)
de_adapted_candidates['source'] = de_adapted_candidates.source.str.replace(
    'dense', 'adapted'
)
de_adapted_candidates.to_parquet(
    de_out / 'cache/candidates_adapted.parquet', index=False
)
pd.DataFrame(de_losses).to_csv(de_out / 'validation/adapter_training.csv', index=False)
for de_source, de_group in de_adapted_candidates[
    de_adapted_candidates.query_id.isin(de_dev.query_id)
].groupby('source'):
    de_pred = predictions_from_frame(
        de_group.sort_values(['query_id', 'rank']), de_items, k=100
    )
    de_result = recall_table(de_pred, de_truth).merge(
        de_dev[['query_id', 'regime']], on='query_id'
    )
    for de_regime, de_part in de_result.groupby('regime'):
        de_scores.append(
            {
                'source': de_source,
                'regime': de_regime,
                **de_part.filter(like='recall@').mean().to_dict(),
            }
        )


In [11]:
pd.DataFrame(de_scores).to_csv(de_out / 'validation/dense_dev.csv', index=False)
display(pd.DataFrame(de_scores))


,source,regime,recall@1,recall@10,recall@20,recall@50,recall@100
0,dense,cold,0.021333,0.111875,0.152792,0.242236,0.304347
1,dense,warm,0.015500,0.089167,0.123667,0.198643,0.271286
2,dense_local,cold,0.181936,0.479744,0.564847,0.630472,0.679014
3,dense_local,warm,0.237833,0.543012,0.617548,0.696012,0.725548
4,adapted,cold,0.016833,0.105542,0.156417,0.243750,0.322861
5,adapted,warm,0.015000,0.077167,0.112667,0.190286,0.273786
6,adapted_local,cold,0.194236,0.491036,0.565014,0.635764,0.682472
7,adapted_local,warm,0.237143,0.552369,0.617238,0.691643,0.729881


## Вывод

Локальный E5 дает 0.63047 на cold и 0.69601 на warm. Адаптер слегка улучшает cold
до 0.63576, но warm снижается до 0.69164. Сам по себе он не лучше во всем.
В 05 проверяю его вместе с текстовыми источниками: ценность источника зависит и от того,
какие объявления он добавляет к уже найденным.